# 03 — Employment and Compensation
## Trade Winds & Headwinds: USVI, 2000–2025

Sections:
1. FRED Leisure & Hospitality employment — monthly SA trend
2. FRED monthly NSA — seasonality
3. FRED annual averages
4. USVIBER tourism-related employment (2010–2022)
5. Census employment snapshots (NAICS 72, 721, 722)
6. Cross-source comparison

**Units note:** FRED values are in thousands of persons. Census employment is a point-in-time count (pay period including March 12). Do not sum FRED monthly values to get annual employment.

In [1]:
import sys, importlib
from pathlib import Path
import os

PROJ_ROOT = Path("..").resolve()
if str(PROJ_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJ_ROOT))
os.chdir(PROJ_ROOT)

import numpy as np
import pandas as pd
import plotly.graph_objects as go

import src.cleaning as _c
import src.charts as _ch

importlib.reload(_c)
importlib.reload(_ch)

from src.cleaning import (
    load_fred_employment,
    load_usviber_tour24,
    load_census_2007,
    load_census_2012_basic,
    load_census_2017_basic,
    load_census_2022_basic,
)
from src.charts import PALETTE, bar_chart, line_chart

pd.set_option("display.float_format", "{:,.1f}".format)
print("Ready.")

Ready.


In [2]:
fred = load_fred_employment()
tour = load_usviber_tour24()
c07 = load_census_2007()
c12 = load_census_2012_basic()
c17 = load_census_2017_basic()
c22 = load_census_2022_basic()
print(f"FRED: {fred.date.min()} – {fred.date.max()}  ({len(fred)} months)")

FRED: 1990-01 – 2026-09  (441 months)


---
## 1. FRED Monthly — Seasonally Adjusted Trend

In [3]:
fred_plot = fred.copy()
fred_plot["date"] = fred_plot["date"].dt.to_timestamp()

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=fred_plot["date"],
        y=fred_plot["emp_sa"],
        mode="lines",
        name="SA employment",
        line=dict(color=PALETTE["employment"], width=2),
    )
)
fig.update_layout(
    title=dict(
        text="USVI Leisure & Hospitality Employment � Seasonally Adjusted<br>"
        "<span style='font-size:13px;color:#555'>Thousands of persons. "
        "Leisure & Hospitality (NAICS 71+72) is broader than tourism-only employment.</span>",
        font=dict(size=17),
    ),
    yaxis_title="Employees (thousands)",
    height=440,
    annotations=[
        dict(
            xref="paper", yref="paper", x=0, y=-0.08,
            text="<span style='color:#888;font-size:11px'>Source: FRED SMS78000007000000001 (BLS)</span>",
            showarrow=False, xanchor="left",
        )
    ],
)
# Historical hurricane and COVID event markers
for x_str, label, color in [
    ("1995-09-01", "Marilyn",      "#777777"),
    ("2017-09-01", "Irma & Maria", "#999999"),
    ("2020-03-01", "COVID-19",     "#999999"),
]:
    fig.add_vline(
        x=pd.Timestamp(x_str),
        line=dict(color=color, width=1.5, dash="dash"),
        annotation_text=label,
        annotation_position="top",
        annotation_font=dict(size=10, color=color),
    )
fig.show()


---
## 2. FRED Monthly — NSA Seasonality

In [4]:
from src.charts import monthly_heatmap

fred_heat = fred.query("year >= 2000").copy()
fred_heat["year_int"] = fred_heat["year"].astype(int)
fred_heat["month_int"] = fred_heat["month"].astype(int)

fig = monthly_heatmap(
    fred_heat,
    year_col="year_int",
    month_col="month_int",
    value_col="emp_nsa",
    title="USVI L&H Employment (NSA) — Seasonal Pattern",
    subtitle="Thousands of persons. Darker = more employment.",
    source="FRED SMU78000007000000001 (BLS)",
    colorscale="Blues",
    height=600,
)
fig.show()

In [5]:
# Average seasonal profile by era
# Bins use right-closed intervals; 2017 gets its own bin as the hurricane shock year.
fred2 = fred.query("year >= 2000").copy()
fred2["era"] = pd.cut(
    fred2["year"],
    bins=[1999, 2016, 2017, 2019, 2021, 2025],
    labels=[
        "Pre-hurricane (2000�2016)",
        "Hurricane shock (2017)",
        "Initial recovery (2018�2019)",
        "COVID shock (2020�2021)",
        "Recent (2022�2025)",
    ],
)
seasonal = (
    fred2.dropna(subset=["emp_nsa", "era"])
    .groupby(["era", "month"])["emp_nsa"]
    .mean()
    .reset_index()
)
month_labels = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec",
]
era_colors = [
    PALETTE["total"],
    "#b22222",
    PALETTE["tourists"],
    PALETTE["cruise"],
    PALETTE["rooms"],
]

fig = go.Figure()
for era, color in zip(seasonal["era"].cat.categories, era_colors):
    sub = seasonal[seasonal["era"] == era].sort_values("month")
    if sub.empty:
        continue
    fig.add_trace(
        go.Scatter(
            x=[month_labels[m - 1] for m in sub["month"]],
            y=sub["emp_nsa"],
            mode="lines+markers",
            name=str(era),
            line=dict(color=color, width=2),
            marker=dict(size=5),
        )
    )

fig.update_layout(
    title=dict(
        text="L&H Employment � Average Seasonal Profile by Era (NSA)<br><span style='font-size:13px;color:#555'>Mean monthly NSA employment (thousands). 2017 shown separately.</span>",
        font=dict(size=17),
    ),
    yaxis_title="Average NSA employment (thousands)",
    height=440,
    annotations=[
        dict(
            xref="paper", yref="paper", x=0, y=-0.08,
            text="<span style='color:#888;font-size:11px'>Source: FRED SMU78000007000000001 (BLS)</span>",
            showarrow=False, xanchor="left",
        )
    ],
)
fig.show()


---
## 3. FRED Annual Averages

In [6]:
# Annual average SA employment (mean of monthly, not sum)
fred_ann = (
    fred.groupby("year")[["emp_sa", "emp_nsa"]]
    .mean()
    .reset_index()
    .rename(columns={"emp_sa": "sa_avg", "emp_nsa": "nsa_avg"})
)

fig = bar_chart(
    fred_ann.query("year >= 2000"),
    x="year",
    y="sa_avg",
    title="USVI L&H Employment — Annual Average (SA)",
    subtitle="Mean of 12 monthly SA observations. Not a sum.",
    y_label="Avg. annual employees (thousands)",
    color=PALETTE["employment"],
    source="FRED SMS78000007000000001 — author calculations",
    shock_x=[2017, 2020],
)
fig.show()

print("Annual average SA employment (thousands):")
print(fred_ann.query("year >= 2000").set_index("year").round(2).to_string())

Annual average SA employment (thousands):
      sa_avg  nsa_avg
year                 
2000     6.8      6.8
2001     6.9      6.9
2002     6.8      6.8
2003     6.9      6.9
2004     7.3      7.3
2005     7.1      7.1
2006     7.2      7.2
2007     7.3      7.3
2008     7.5      7.5
2009     6.9      6.9
2010     7.2      7.2
2011     7.4      7.4
2012     7.3      7.3
2013     7.2      7.2
2014     7.2      7.2
2015     7.4      7.4
2016     7.7      7.7
2017     6.9      6.9
2018     4.2      4.2
2019     5.6      5.6
2020     4.9      4.9
2021     5.8      5.8
2022     6.3      6.3
2023     6.8      6.8
2024     7.0      7.0
2025     6.5      6.5
2026     5.5      5.5


---
## 4. USVIBER Tourism-Related Employment

In [7]:
# USVIBER employment — available 2010–2022 only
tour_emp = tour[
    ["year", "emp_tourism_total", "emp_hotels", "emp_eating_drinking"]
].dropna(subset=["emp_tourism_total"])

fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=tour_emp["year"],
        y=tour_emp["emp_hotels"],
        name="Hotels & lodging",
        marker_color=PALETTE["tourists"],
    )
)
fig.add_trace(
    go.Bar(
        x=tour_emp["year"],
        y=tour_emp["emp_eating_drinking"],
        name="Eating & drinking places",
        marker_color=PALETTE["cruise"],
    )
)
fig.update_layout(
    barmode="stack",
    title=dict(
        text="USVIBER Tourism-Related Employment (Selected Categories)<br>"
        '<span style="font-size:13px;color:#555">Number of jobs. Data ends 2022; 2023–2024 not available from DOL.</span>',
        font=dict(size=17),
    ),
    yaxis_title="Number of jobs",
    yaxis_tickformat=",",
    height=440,
    annotations=[
        dict(
            xref="paper",
            yref="paper",
            x=0,
            y=-0.08,
            text='<span style="color:#888;font-size:11px">Source: USVIBER Annual Tourism Indicator 2024 [a] U.S. Virgin Islands Bureau of Labor Statistics</span>',
            showarrow=False,
            xanchor="left",
        )
    ],
)
for x_val, label in [(2017, "Irma & Maria"), (2020, "COVID-19")]:
    fig.add_vline(
        x=x_val,
        line=dict(color="#999", width=1.5, dash="dash"),
        annotation_text=label,
        annotation_position="top",
        annotation_font=dict(size=10, color="#666"),
    )
fig.show()
print(tour_emp.set_index("year").to_string())

      emp_tourism_total  emp_hotels  emp_eating_drinking
year                                                    
2010            8,301.0     3,587.0              2,793.0
2011            8,469.0     3,549.0              2,959.0
2012            8,339.0     3,617.0              2,821.0
2013            8,136.0     3,519.0              2,784.0
2014            8,171.0     3,575.0              2,809.0
2015            8,159.0     3,629.0              2,765.0
2016            8,430.0     3,617.0              3,170.0
2017            7,332.0     3,329.0              2,765.0
2018            5,093.0     1,559.0              2,420.0
2019            6,138.0     1,784.0              3,039.0
2020            5,315.0     1,664.0              2,667.0
2021            6,221.0     2,131.0              3,009.0
2022            6,819.0     2,448.0              3,075.0


---
## 5. Census Employment Snapshots

In [8]:
# Build a comparable Census NAICS 72 / 721 / 722 table across vintages
frames = []
for df, vintage_year in [(c07, 2007), (c12, 2012), (c17, 2017), (c22, 2022)]:
    sub = df[df["naics"].astype(str).isin(["72", "721", "722"])][
        ["year", "naics", "naics_label", "estab", "rcptot_k", "payann_k", "emp"]
    ].copy()
    sub["vintage"] = vintage_year
    frames.append(sub)

census = pd.concat(frames, ignore_index=True)
# De-duplicate: each vintage file contains the prior period's data too;
# keep only the row where year matches the vintage for each file
# (or if a file contains two years, keep both)
census_key = census[census["naics"] == "72"].sort_values("year")

print("=== Census NAICS 72: Accommodation & Food Services ===")
print(
    census_key[["year", "estab", "emp", "payann_k", "rcptot_k"]]
    .drop_duplicates()
    .set_index("year")
    .to_string()
)

=== Census NAICS 72: Accommodation & Food Services ===
      estab     emp  payann_k  rcptot_k
year                                   
2002    313 5,639.0  92,357.0 331,008.0
2007    255 6,146.0 129,635.0 460,532.0
2012    279 6,864.0 160,542.0 539,442.0
2017    304 6,737.0 149,735.0 520,417.0
2022    293 5,140.0 136,885.0 536,134.0


In [9]:
# Plot Census employment for NAICS 72, 721, 722
# Explicitly prefer the later vintage when years overlap across vintage files:
# year 2012 appears in both vintage-2012 and vintage-2017 files; keep vintage-2012.
# year 2017 appears in both vintage-2017 and vintage-2022 files; keep vintage-2017.
census_dedup = (
    census
    .sort_values(["year", "vintage"])          # earlier vintage first
    .drop_duplicates(subset=["year", "naics"], keep="first")  # keep earlier vintage
)

fig = go.Figure()
for naics_code, name, color in [
    ("72",  "Accommodation & food services (NAICS 72)",  PALETTE["total"]),
    ("721", "Accommodation (NAICS 721)",                  PALETTE["tourists"]),
    ("722", "Food services & drinking places (NAICS 722)",PALETTE["cruise"]),
]:
    sub = (
        census_dedup[census_dedup["naics"] == naics_code]
        .dropna(subset=["emp"])
        .sort_values("year")
    )
    fig.add_trace(
        go.Scatter(
            x=sub["year"],
            y=sub["emp"],
            mode="markers+lines",
            name=name,
            line=dict(color=color, width=2, dash="dot"),
            marker=dict(size=10, symbol="diamond"),
        )
    )

fig.update_layout(
    title=dict(
        text="Census Employment Snapshots � Accommodation & Food Services<br>"
        "<span style='font-size:13px;color:#555'>Point-in-time (March pay period). Quinquennial snapshots; not a continuous series.</span>",
        font=dict(size=17),
    ),
    yaxis_title="Paid employees (point-in-time)",
    yaxis_tickformat=",",
    height=440,
    annotations=[
        dict(
            xref="paper", yref="paper", x=0, y=-0.08,
            text="<span style='color:#888;font-size:11px'>Source: U.S. Census Economic Census 2007, 2012, 2017, 2022 � USVI territory level. 2017 reference date = March 2017 pay period.</span>",
            showarrow=False, xanchor="left",
        )
    ],
)
for x_val, label in [(2017, "Irma & Maria"), (2020, "COVID-19")]:
    fig.add_vline(
        x=x_val,
        line=dict(color="#999", width=1.5, dash="dash"),
        annotation_text=label,
        annotation_position="top",
        annotation_font=dict(size=10, color="#666"),
    )
fig.show()


In [10]:
# Payroll and receipts (nominal)
c_n72 = (
    census_dedup[census_dedup["naics"] == "72"]
    .dropna(subset=["payann_k"])
    .sort_values("year")
)

fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=c_n72["year"],
        y=c_n72["rcptot_k"] / 1000,
        name="Receipts ($M)",
        marker_color=PALETTE["total"],
    )
)
fig.add_trace(
    go.Bar(
        x=c_n72["year"],
        y=c_n72["payann_k"] / 1000,
        name="Annual payroll ($M)",
        marker_color=PALETTE["tourists"],
    )
)
fig.update_layout(
    barmode="group",
    title=dict(
        text="NAICS 72 — Receipts and Payroll (Nominal $M)<br>"
        '<span style="font-size:13px;color:#555">Census snapshots. No inflation adjustment.</span>',
        font=dict(size=17),
    ),
    yaxis_title="$M (nominal)",
    height=420,
    annotations=[
        dict(
            xref="paper",
            yref="paper",
            x=0,
            y=-0.08,
            text='<span style="color:#888;font-size:11px">Source: U.S. Census Economic Census 2007–2022 — USVI territory level</span>',
            showarrow=False,
            xanchor="left",
        )
    ],
)
fig.show()

---
## 6. Cross-Source Comparison

Three employment series measure overlapping but distinct things:

| Series | Scope | Frequency | Units | Reference |
|---|---|---|---|---|
| FRED SA/NSA | Leisure & Hospitality (NAICS 71+72) | Monthly | Thousands | Pay-period average |
| USVIBER | Tourism-related (hotels, eating, transport, gift) | Annual | Jobs | Year-end survey |
| Census NAICS 72 | Accommodation & food services only | Quinquennial | Jobs | March pay period |

**Do not equate these series.** FRED is broadest (includes arts/recreation). Census NAICS 72 excludes transportation and retail. USVIBER's category list differs from standard NAICS.

In [11]:
# Side-by-side comparison for years with overlapping data
fred_ann = (
    fred.query("year >= 2000")
    .groupby("year")["emp_sa"]
    .mean()
    .mul(1000)  # convert to absolute count for comparison
    .reset_index()
    .rename(columns={"emp_sa": "fred_sa_avg"})
)
usviber_emp = tour[["year", "emp_tourism_total"]].dropna()
census_72 = (
    census_dedup[census_dedup["naics"] == "72"][["year", "emp"]]
    .dropna()
    .rename(columns={"emp": "census_n72"})
)

comp = (
    fred_ann.merge(usviber_emp, on="year", how="outer")
    .merge(census_72, on="year", how="outer")
    .sort_values("year")
)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=comp["year"],
        y=comp["fred_sa_avg"],
        mode="lines",
        name="FRED L&H (SA annual avg)",
        line=dict(color=PALETTE["total"], width=2),
    )
)
fig.add_trace(
    go.Scatter(
        x=comp["year"],
        y=comp["emp_tourism_total"],
        mode="lines+markers",
        name="USVIBER tourism employment",
        line=dict(color=PALETTE["tourists"], width=2),
    )
)
fig.add_trace(
    go.Scatter(
        x=comp["year"],
        y=comp["census_n72"],
        mode="markers",
        name="Census NAICS 72 (March)",
        marker=dict(color=PALETTE["cruise"], size=12, symbol="diamond"),
    )
)

fig.update_layout(
    title=dict(
        text="Employment Cross-Source Comparison<br>"
        '<span style="font-size:13px;color:#555">Three different series with different scopes — do not assume equivalence</span>',
        font=dict(size=17),
    ),
    yaxis_title="Employees",
    yaxis_tickformat=",",
    height=460,
    annotations=[
        dict(
            xref="paper",
            yref="paper",
            x=0,
            y=-0.08,
            text='<span style="color:#888;font-size:11px">Sources: FRED, USVIBER, U.S. Census Economic Census</span>',
            showarrow=False,
            xanchor="left",
        )
    ],
)
for x_val, label in [(2017, "Irma & Maria"), (2020, "COVID-19")]:
    fig.add_vline(
        x=x_val,
        line=dict(color="#999", width=1.5, dash="dash"),
        annotation_text=label,
        annotation_position="top",
        annotation_font=dict(size=10, color="#666"),
    )
fig.show()
print(comp.set_index("year").round(0).to_string())

      fred_sa_avg  emp_tourism_total  census_n72
year                                            
2000      6,800.0                NaN         NaN
2001      6,917.0                NaN         NaN
2002      6,783.0                NaN     5,639.0
2003      6,900.0                NaN         NaN
2004      7,283.0                NaN         NaN
2005      7,117.0                NaN         NaN
2006      7,150.0                NaN         NaN
2007      7,342.0                NaN     6,146.0
2008      7,525.0                NaN         NaN
2009      6,933.0                NaN         NaN
2010      7,233.0            8,301.0         NaN
2011      7,358.0            8,469.0         NaN
2012      7,308.0            8,339.0     6,864.0
2013      7,158.0            8,136.0         NaN
2014      7,250.0            8,171.0         NaN
2015      7,400.0            8,159.0         NaN
2016      7,692.0            8,430.0         NaN
2017      6,917.0            7,332.0     6,737.0
2018      4,233.0   